# A Gentle Introduction to `torch.audograd`
`torch.autograd` is PyTorch's automatic differentiation engine that powers neural network training.

## Background
Neural Networks (NNs) are an collection of *nested functions* that are computed on some input data. These functions are defined by *parameters* (weights adn biases), which in PyTorch are stored in tensors.

Training of a NN is executed internally in two steps:

-**Forward Pass**- The input value is computed through each of its functions to make the best guess about the correct output.
-**Backward Pass**- NN adjusts its parameters proportionate to the error in its guess. It does this by traversing backwards from the output, collecting the derivatives of the error wrt the parameters of the functions (*gradients*), and optimizing the parameters using gradient descent. Later watch a walkthrough of **backprop** with a [video from 3Blue 1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8)

## Usage in PyTorch

A single training step has been shown below. First, we load a pretrained resnet18 model from `torchvision`. A random data vector is initialized to represent a single image with 3 channels, and height and width of 64, and its corresponding `label` initialized to some random values. Label in pretrained models has shape (1, 1000).

In [2]:
import torch
from torchvision.models import resnet18, ResNet18_Weights



In [3]:
device = 'cpu'
model = resnet18(weights=ResNet18_Weights.DEFAULT).to(device)
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

Next, run the input data through the model i.e. through each of its layers to make a prediction. This is the **forward pass.**

In [25]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Just printing the model is a bad idea to visualize the model. Use `summary` from `torchinfo`.

In [26]:
from torchinfo import summary
summary(model, input_size=(1, 3, 64, 64))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 32, 32]           9,408
├─BatchNorm2d: 1-2                       [1, 64, 32, 32]           128
├─ReLU: 1-3                              [1, 64, 32, 32]           --
├─MaxPool2d: 1-4                         [1, 64, 16, 16]           --
├─Sequential: 1-5                        [1, 64, 16, 16]           --
│    └─BasicBlock: 2-1                   [1, 64, 16, 16]           --
│    │    └─Conv2d: 3-1                  [1, 64, 16, 16]           36,864
│    │    └─BatchNorm2d: 3-2             [1, 64, 16, 16]           128
│    │    └─ReLU: 3-3                    [1, 64, 16, 16]           --
│    │    └─Conv2d: 3-4                  [1, 64, 16, 16]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 16, 16]           128
│    │    └─ReLU: 3-6                    [1, 64, 16, 16]           --
│

In [ ]:
prediction = model(data) # forward pass


List of all attributes and methods: ['H', 'T', '__abs__', '__add__', '__and__', '__annotations__', '__array__', '__array_priority__', '__array_wrap__', '__bool__', '__class__', '__complex__', '__contains__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__div__', '__dlpack__', '__dlpack_c_exchange_api__', '__dlpack_device__', '__doc__', '__eq__', '__float__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__idiv__', '__ifloordiv__', '__ilshift__', '__imod__', '__imul__', '__index__', '__init__', '__init_subclass__', '__int__', '__invert__', '__ior__', '__ipow__', '__irshift__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__long__', '__lshift__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__nonzero__', '__or__', '__pos__', '__pow__', '__radd__', '__rand__', '__rdiv__', '__reduce__', '__reduce_ex

Use model's prediction and the corresponding label to calculate the error(`loss`). Then backpropagate this error through the network. Backward propagation is kicked off when we call `.backward()` on the error tensor. Autograd calculates and stores the gradients for each model parameter in the parameter's `.grad` attribute.

In [32]:
loss = (prediction - labels).sum()
loss.backward() # backward pass

Next, load an SGD optimizer with a learning rate of 0.01 and momentum of 0.9. Register all the parameters of the model in the optimizer.

In [33]:
optim = torch.optim.SGD(model.parameters(), lr = 1e-2, momentum=0.9)

Finally, call `step()` to initiate gradient descent.

In [34]:
optim.step()

Understanding of the above steps will be sufficient for training of the neural network.

#### Differentiation in Autograd
Let's create two tensors `a` and `b` with `requires_grad`. This signals to `autograd` that every operation on them should be tracked.

In [ ]:
a = torch.tensor([2., 3.], requires_grad= True)
b = torch.tensor([6., 4.], requires_grad=True)
print(a)

tensor([2., 3.], requires_grad=True)


Create another tensor `Q` from `a` and `b`.
$$
\begin{equation}
Q= 3a^3 - b^2
\end{equation}
$$

In [ ]:
Q = 3*a**3 -b**2
print(Q)

tensor([-12.,  65.], grad_fn=<SubBackward0>)


Assuming that `a` and `b` are parameters of an NN, and `Q` to be the error. In NN training, we want gradients of the error w.r.t. parameters, i.e.

$$
\begin{aligned}
\frac{\partial Q}{\partial a} &= 9 a^2 \\
\frac{\partial Q}{\partial b} &= -2b
\end{aligned}
$$

Call `.backward()` on `Q`, autograd calculates these gradients and stores them in the respective tensors `.grad` attribute.
To understand vector external_grad $ v = [1 \quad 1]$, consider the operation
$$
\begin{aligned}
v^T \cdot J &= \begin{bmatrix} 1 & 1 \end{bmatrix} \cdot \begin{pmatrix}\frac{\partial Q_1}{\partial a_1} &  \frac{\partial Q_1}{\partial a_2} \\ 
\frac{\partial Q_2}{\partial a_1} & \frac{\partial Q_2}{\partial a_2} 
\end{pmatrix} \\
&=  \begin{bmatrix} 1 & 1 \end{bmatrix} \cdot  \begin{pmatrix} 9 \cdot a_1^2 & 0  \\ 
0 & 9 \cdot a_2^2 
\end{pmatrix} \\

&= \begin{bmatrix} 9 \cdot a_1^2 & 9 \cdot a_2^2 \end{bmatrix} \\

\text{where} \\
Q_1 &= 3 \cdot a_1^3 - b_1^2 \\
Q_2 &= 3 \cdot a_2^3 - b_2^2
\end{aligned}
$$
that gets computed in `.backward()` routine for a and similarly for $b$.

**Back Propagation Algorithm:** Backprop algorithm is the engine behind training NNs using labelled data. It efficiently performs the computation of differentiation operator of heavily nested functions. The idea of backprop is to first decouple partial computation into local gradient computation: $J_{z_{in}}$ and $J_{z_{W}}$ , computed during forward pass and into upstream gradient ($v$), computed during backward pass. And finally coupling them together during backward gradient computation and update of weights.
Each layer only needs to know two things:
1. $J$ aka local gradient &ndash; How does the current layer's output change wrt to inputs? &ndash; computed during the forward pass and stored in the computation graph.
2. $v$ aka upstream gradient &ndash; gradient flowing in from the layer ahead (closer to the loss).
```
loss = 1/2(y - z_3)^2

Forward:  input → z1 → z2 → z3 → loss
Backward: 
  seed     :  dL/dL = 1       (trivial seed , seed = 1, internal to PyTorch)
  Layer out:  v = dL/dz3 = -(y - z3), ← first real v  
  Layer3   :  v = -(y - z3),          a.grad = v · J3  →  passes dL/dz2 back
  Layer2   :  v = dL/dz2,     a.grad = v · J2  →  passes dL/dz1 back
  Layer1   :  v = dL/dz1,     a.grad = v · J1  →  passes dL/dinput back

```
That is why PyTorch stores `grad_fn` on every tensor &ndash; each `grad_fn` knows how to compute its local $J$. And apply the Vector Jacobian Product (VJP) $v^T \times J$ when `backward()` walks the graph.

3. The local jacobian has two parts:
   1. $\frac{\partial z_{out}}{\partial W}$ - wrt weights (used to update weights Via gradient descent).
   2. $\frac{\partial z_{out}}{\partial z_{in}}$ - wrt the layer's input (used to pass the gradient further back to the previous layer).

So, at each layer two VJP are computed from the same gradient upstream ($v$).
$$
\begin{aligned}
J_W = \frac{\partial L}{\partial W} &= v \cdot \frac{\partial z_{out}}{\partial W} \quad \leftarrow \text{stored in W.grad, used in `optim.step()` by optimizer to update weights} \\
J_{z_{in}} = \frac{\partial L}{\partial z_{in}} &= v \cdot \frac{\partial z_{out}}{\partial z_{in}} \quad \leftarrow \text{becomes the new v passed to the previous layer}\\
\end{aligned}
$$

In [57]:
external_grad = torch.tensor([1.,1.])
Q.backward(gradient=external_grad)

In [ ]:
print(f"9*a^2 = {a.grad}")
print(f"-2*b = {b.grad}")

9*a^2 = tensor([36., 81.])
-2*b = tensor([-12.,  -8.])


### Vector Calculus using `autograd`

Generally speaking, `torch.autograd` is an engine for computing vector-jacobian product. That is, given any vector $\vec{v}$, compute the product $J^T \cdot \vec{v}$. Consider a vector valued function $\vec{y} = g(\vec{x})$:
$$
\begin{aligned}
\frac{\partial \vec{y}}{\partial \vec{x}} &= \begin{pmatrix} \frac{\partial \vec{y}}{\partial x_1} & \cdots &\frac{\partial \vec{y}}{\partial x_n}    \end{pmatrix} & = \begin{bmatrix} \frac{\partial y_1}{\partial x_1} & \cdots &\frac{\partial y_1}{\partial x_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial y_n}{\partial x_1} & \cdots &\frac{\partial y_n}{\partial x_n}\end{bmatrix}
\end{aligned}
$$

If $\vec{v}$ is gradient of a scalar function $ l = g(\vec{y})$.
$$
\begin{aligned}
\vec{v} = \begin{pmatrix} \frac{\partial l}{\partial y_1}& \cdots & \frac{\partial l}{\partial y_n}\end{pmatrix}^T
\end{aligned}
$$
Then,
$$
\begin{aligned}
J^T \cdot \vec{v} &= \begin{bmatrix} \frac{\partial y_1}{\partial x_1} & \cdots &\frac{\partial y_1}{\partial x_1} \\ \vdots & \ddots & \vdots \\ \frac{\partial y_n}{\partial x_n} & \cdots &\frac{\partial y_n}{\partial x_n}\end{bmatrix} \cdot \begin{pmatrix} \frac{\partial l}{\partial y_1} \\ \vdots \\ \frac{\partial l}{\partial y_n}\end{pmatrix}
\end{aligned}
$$

#### Computational Graph
Autograd keeps a record of data (tensors) and all executed operations (along with resulting new tensors) in a directed acyclic graph (DAG). In DAG, leaves are input tensors and roots are output tensors. By tracing this graph from roots to leaves, gradients can be computed automatically.
In the forward pass, autograd does two things simultaneously:
* Run the requested operation to compute a resulting tensor.
* Maintain the operations *gradient function* in the DAG.
For backward pass, `.backward()` is called on the DAG root. `autograd` then:
* computes the gradients from each `.grad_fn`,
* accumulates them in respective tensor's `.grad` attribute, and 
* using the chain rule, propagate s all the way to the leaf tensors.

**DAG for `Q = 3*a**3 - b**2`**

**Forward Pass** — data flows leaf → root (top to bottom):

```mermaid
flowchart TB
    a("a = [2., 3.]\nleaf · requires_grad=True")
    b("b = [6., 4.]\nleaf · requires_grad=True")
    pow3["power: a**3\ngrad_fn stores: 3a²"]
    mul3["multiply: *3\ngrad_fn stores: 3"]
    pow2["power: b**2\ngrad_fn stores: 2b"]
    Q["subtract  ← ROOT\nQ = 3a³ − b²"]

    a -->|"a³"| pow3
    pow3 -->|"a³"| mul3
    mul3 -->|"3a³"| Q
    b -->|"b²"| pow2
    pow2 -->|"b²"| Q

    style a fill:#90EE90,color:#000
    style b fill:#90EE90,color:#000
    style Q fill:#ff9999,color:#000
```

**Backward Pass** — gradients flow root → leaf when `Q.backward(v)` is called:

```mermaid
flowchart BT
    a("a.grad = 9a² = [36., 27.]")
    b("b.grad = −2b = [−12., −8.]")
    pow3["power: a**3\npasses v·3a² back"]
    mul3["multiply: *3\npasses v·3 back"]
    pow2["power: b**2\npasses v·2b back"]
    Q["ROOT Q\nseed v = [1, 1]"]

    pow3 -->|"v·3a² = 9a²"| a
    mul3 -->|"v·3"| pow3
    Q -->|"v = +[1,1]"| mul3
    pow2 -->|"v·2b = −2b"| b
    Q -->|"v = −[1,1]"| pow2

    style a fill:#90EE90,color:#000
    style b fill:#90EE90,color:#000
    style Q fill:#ff9999,color:#000
```

> Green nodes = **leaves** (`.grad` stored here). Red node = **root** (`.backward()` called here). Intermediate nodes only hold a `grad_fn` and pass gradients through.

#### Exclusion from DAG
The tensors which does not require gradient computation, the `tensor.requires_grad` attribute can be set to `False`. These are usually called **frozen parameters**.
This is specially important in case of finetuning, when we freeze most of the model and only modify the classifier layer to make predictions on new labels. 

In [5]:
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# freeze all the parameters in the network
for param in model.parameters():
    param.requires_grad = False
    

Now, let's finetune the model on a new dataset with $10$ labels. Classifier is the last linearlayer `model.fc` in resnet. It can be replaced with a new linear layer (which is unfrozen by default) that acts as a classifier.

In [6]:
model.fc = nn.Linear(512, 10)

All the parameters in the model, except the parameters of the last layer are frozen. The only parameters that computes the gradient are weights and bias of `model.fc`.

In [7]:
# optimize only the classifier
optimizer = optim.SGD(model.parameters(), lr= 1e-2, momentum=0.9)

Note that although all the parameters are registered in the optimizer. The only parameters for which gradient is computed (and hence updated in gradient descent) are weights and bias. The same exclusionary functionality which is available as a context manager is ``torch.no_grad()``

### Loss function as stochastic optimization problem
#### Scientific ML Lecture 24.4

The optimization problem can be highly non-linear in case of DNN with many local minima. Stochastic optimization algorithms tend to work better than deterministic optimization algorithms.
Take for example regression task and define Mean Square Error Loss Function.
Data points:    &nbsp; &nbsp;$ x_1 \cdots x_n$.  
Target points:  &nbsp; &nbsp; $y_1 \cdots y_n$.  
DNN output: &nbsp; &nbsp; $f(x_i; \theta)$.  
DNN loss function: &nbsp; &nbsp; $L(\theta) = \frac{1}{n} \sum_{i=1}^{n} [\ (y_i - f(x_i))^2 ]\  $  
Define a categorical variable:&nbsp; &nbsp; $I \sim Categorical(1/n, 1/n, \cdots, 1/n)$.  
Define a loss function as a function of $I$: $l(\theta; I) = [\ (y_I -f(x_I; \theta))^2]\  $
$$
\begin{aligned}
\mathbb{E}_I[ l(\theta; I)] &= \mathbb{E}_I [\ (y_I -f(x_I; \theta))^2]\  \\
&= \sum_{i=0}^{n} P(I=i) \times [\ (y_i -f(x_i; \theta))^2]\  \\
&= \sum_{i=0}^{n} 1/n \times [\ (y_i -f(x_i; \theta))^2]\  \\
&= \frac{1}{n} \sum_{i=0}^{n} \times [\ (y_i -f(x_i; \theta))^2]\  \\
&= L(\theta)
\end{aligned}
$$

Hence, the optimization problem can be reformulated in stochastic domain as: &nbsp; $ \min_\theta \mathbb{E}_I [\ (y_I -f(x_I; \theta))^2]\ $  &nbsp; Where $I$ is a categorical random variable taking values in $\{1, 2, \cdots, n\}$ with equal probability.  
Equivalently, optimization problem can be reformulated in stochastic domain as: &nbsp; $ \min_\theta \mathbb{E}_I [\ 0.5(y_{I_1} -f(x_{I_1}; \theta))^2 + 0.5(y_{I_1} -f(x_{I_1}; \theta))^2]\ $  &nbsp; Where $I_1$ and $I_2$ are independent categorical random variable taking values in $\{1, 2, \cdots, n\}$ with equal probability.  
The optimization problem can be reformulated in stochastic domain further for batch processing as: &nbsp; $ \min_\theta \mathbb{E}_I [\ \sum_{j=0}^m \frac{1}{m} \times (y_{I_j} -f(x_{I_j}; \theta))^2]\ $  &nbsp; Where $I_j$ is a categorical random variable for $j \in \{1, 2, \cdots , m \}$  taking values in $\{1, 2, \cdots, n\}$ with equal probability. 

#### Scientific ML Lecture 24.5

The Robbins-Monro (RM) algorithm is the simplest Stochastic gradient algorithm.The optimization problem can be written as: $$ \min_\theta \mathbb{E}_z [\ l(\theta; z)]\ (*)$$  
The optimization algorithm can be described as follows:
* Initialize $\theta_0$.
* Iterate: $$ \theta_{t+1} = \theta_t - \alpha_t \nabla_\theta l(\theta; z_t) $$ Where $z_t$ is a sample of $z$. $\alpha_t$ is the learning rate.  
The RM algorithm states that the algorithm converges to a local minimum of the stochastic optimization problem $(*)$. if $$\sum_{t=1}^\infty \alpha_t = +\infty  ; \sum_{\alpha_t}^\infty \alpha_t^2 < +\infty$$ Meaning $\alpha_t \to 0$ but not too fast.
* $\alpha_t =\frac{A}{(Bt + C)^\rho}$ where $0.5< \rho <1$ 
